In [31]:
import sys
import os
import pandas as pd
from sqlalchemy import create_engine, text

# Add parent directory to path (notebook is in scripts folder)
notebook_dir = os.getcwd() if 'scripts' in os.getcwd() else os.path.join(os.getcwd(), 'chatbot-backend', 'scripts')
parent_dir = os.path.dirname(notebook_dir)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from app.database.session import get_db
from app.database.models import (
    User, Conversation, Message, Question, Answer,
    Topic, Subtopic, TopicDependency, DocumentChunk
)

# Get database connection
db = next(get_db())
engine = db.bind

# Load all tables as pandas DataFrames
print("Loading tables...")

users_df = pd.read_sql_table('users', engine)
print(f"✓ Users: {len(users_df)} rows")

conversations_df = pd.read_sql_table('conversations', engine)
print(f"✓ Conversations: {len(conversations_df)} rows")

messages_df = pd.read_sql_table('messages', engine)
print(f"✓ Messages: {len(messages_df)} rows")

questions_df = pd.read_sql_table('questions', engine)
print(f"✓ Questions: {len(questions_df)} rows")

answers_df = pd.read_sql_table('answers', engine)
print(f"✓ Answers: {len(answers_df)} rows")

topics_df = pd.read_sql_table('topics', engine)
print(f"✓ Topics: {len(topics_df)} rows")

subtopics_df = pd.read_sql_table('subtopics', engine)
print(f"✓ Subtopics: {len(subtopics_df)} rows")

topic_dependencies_df = pd.read_sql_table('topic_dependencies', engine)
print(f"✓ Topic Dependencies: {len(topic_dependencies_df)} rows")

question_topics_df = pd.read_sql_table('question_topics', engine)
print(f"✓ Question Topics: {len(question_topics_df)} rows")

question_subtopics_df = pd.read_sql_table('question_subtopics', engine)
print(f"✓ Question Subtopics: {len(question_subtopics_df)} rows")

print("\n✓ All tables loaded successfully!")


Loading tables...
✓ Users: 5 rows
✓ Conversations: 45 rows
✓ Messages: 398 rows
✓ Questions: 199 rows
✓ Answers: 199 rows
✓ Topics: 6 rows
✓ Subtopics: 29 rows
✓ Topic Dependencies: 6 rows
✓ Question Topics: 405 rows
✓ Question Subtopics: 804 rows

✓ All tables loaded successfully!


In [32]:
# Analyze user 202's conversations
user_id = 202
user_conversations = conversations_df[conversations_df['user_id'] == user_id].copy()

if len(user_conversations) > 0:
    print(f"User {user_id} has {len(user_conversations)} conversations")
    print(f"\nConversation titles:")
    print(user_conversations[['id', 'title', 'last_accessed']].to_string(index=False))
    
    # Get message statistics for each conversation
    print(f"\n\nMessage analysis per conversation:")
    for conv_id in user_conversations['id']:
        conv_messages = messages_df[messages_df['conversation_id'] == conv_id].copy()
        if len(conv_messages) > 0:
            conv_messages['timestamp'] = pd.to_datetime(conv_messages['timestamp'])
            first_msg = conv_messages['timestamp'].min()
            last_msg = conv_messages['timestamp'].max()
            duration_minutes = (last_msg - first_msg).total_seconds() / 60
            
            conv_title = user_conversations[user_conversations['id'] == conv_id]['title'].values[0]
            print(f"\nConversation {conv_id}: {conv_title}")
            print(f"  Messages: {len(conv_messages)}")
            print(f"  Duration: {duration_minutes:.2f} minutes")
            print(f"  First message: {first_msg}")
            print(f"  Last message: {last_msg}")
else:
    print(f"No conversations found for user {user_id}")



User 202 has 9 conversations

Conversation titles:
 id                  title                    last_accessed
296 Day 1 - Conversation 1 2026-01-15 02:47:30.107878+00:00
297 Day 2 - Conversation 1 2026-01-16 02:49:59.107878+00:00
298 Day 3 - Conversation 1 2026-01-17 11:30:44.107878+00:00
299 Day 4 - Conversation 1 2026-01-18 14:12:18.107878+00:00
300 Day 5 - Conversation 1 2026-01-19 08:53:32.107878+00:00
301 Day 6 - Conversation 1 2026-01-20 13:36:56.107878+00:00
302 Day 7 - Conversation 1 2026-01-21 07:44:23.107878+00:00
303 Day 8 - Conversation 1 2026-01-22 04:07:33.107878+00:00
304 Day 9 - Conversation 1 2026-01-23 09:20:50.107878+00:00


Message analysis per conversation:

Conversation 296: Day 1 - Conversation 1
  Messages: 10
  Duration: 26.37 minutes
  First message: 2026-01-15 02:18:50.107878+00:00
  Last message: 2026-01-15 02:45:12.107878+00:00

Conversation 297: Day 2 - Conversation 1
  Messages: 6
  Duration: 14.93 minutes
  First message: 2026-01-16 02:31:53.107878+00:0

In [33]:
# print first and last messages for each of the conversations of user 202, and their IDs
for conv_id in user_conversations['id']:
    conv_messages = messages_df[messages_df['conversation_id'] == conv_id].copy()
    if len(conv_messages) > 0:
        conv_messages = conv_messages.sort_values('timestamp')
        conv_title = user_conversations[user_conversations['id'] == conv_id]['title'].values[0]
        
        first_msg = conv_messages.iloc[0]
        last_msg = conv_messages.iloc[-1]
        
        print(f"\n{'='*80}")
        print(f"Conversation {conv_id}: {conv_title}")
        print(f"{'='*80}")
        print(f"\nFirst Message (ID: {first_msg['id']}):")
        print(f"  Sender: {first_msg['sender']}")
        print(f"  Timestamp: {first_msg['timestamp']}")
        print(f"  Content: {first_msg['content'][:200]}{'...' if len(first_msg['content']) > 200 else ''}")
        
        print(f"\nLast Message (ID: {last_msg['id']}):")
        print(f"  Sender: {last_msg['sender']}")
        print(f"  Timestamp: {last_msg['timestamp']}")
        print(f"  Content: {last_msg['content'][:200]}{'...' if len(last_msg['content']) > 200 else ''}")



Conversation 296: Day 1 - Conversation 1

First Message (ID: 2566):
  Sender: user
  Timestamp: 2026-01-15 02:18:50.107878+00:00
  Content: User question 1: What is the concept?

Last Message (ID: 2575):
  Sender: assistant
  Timestamp: 2026-01-15 02:45:12.107878+00:00
  Content: Assistant answer 5: Here is the explanation...

Conversation 297: Day 2 - Conversation 1

First Message (ID: 2576):
  Sender: user
  Timestamp: 2026-01-16 02:31:53.107878+00:00
  Content: User question 1: What is the concept?

Last Message (ID: 2581):
  Sender: assistant
  Timestamp: 2026-01-16 02:46:49.107878+00:00
  Content: Assistant answer 3: Here is the explanation...

Conversation 298: Day 3 - Conversation 1

First Message (ID: 2582):
  Sender: user
  Timestamp: 2026-01-17 11:09:50.107878+00:00
  Content: User question 1: What is the concept?

Last Message (ID: 2589):
  Sender: assistant
  Timestamp: 2026-01-17 11:27:19.107878+00:00
  Content: Assistant answer 4: Here is the explanation...

Conversation

In [34]:
# ccompare user 202's interaction counts with user 201's
user_ids = [201, 202]
for uid in user_ids:
    user_convs = conversations_df[conversations_df['user_id'] == uid]
    total_messages = 0
    for conv_id in user_convs['id']:
        conv_msgs = messages_df[messages_df['conversation_id'] == conv_id]
        total_messages += len(conv_msgs)
    print(f"User {uid} has {len(user_convs)} conversations with a total of {total_messages} messages.")

User 201 has 9 conversations with a total of 82 messages.
User 202 has 9 conversations with a total of 76 messages.


In [35]:
# average interaction count of everyone
total_users = len(users_df)
total_messages = len(messages_df)
average_messages_per_user = total_messages / total_users if total_users > 0 else 0
print(f"\nAverage messages per user: {average_messages_per_user:.2f}")


Average messages per user: 79.60


In [36]:
# sort number of questions WITH answers asked by topic
# First get questions that have answers
answered_question_ids = answers_df['question_id'].unique()
answered_questions_topics = question_topics_df[question_topics_df['question_id'].isin(answered_question_ids)]

# Count by topic
question_topic_counts = answered_questions_topics['topic_id'].value_counts().reset_index()
question_topic_counts.columns = ['topic_id', 'count']

# Merge with topic names
question_topic_counts = question_topic_counts.merge(topics_df[['id', 'topic_name']], left_on='topic_id', right_on='id')
question_topic_counts = question_topic_counts[['topic_name', 'count']].sort_values('count', ascending=False)

print("Questions WITH answers by topic:")
print(question_topic_counts.to_string(index=False))
print(f"\nTotal answered questions: {len(answered_questions_topics)}")


Questions WITH answers by topic:
                                                    topic_name  count
                                      Transfer Function Models     73
                      Theoretical Models of Chemical Processes     70
   Dynamic Behaviour of First-Order and Second-Order Processes     68
                               Introduction to Process Control     66
Dynamic Response Characteristics of More Complicated Processes     65
                                            Laplace Transforms     63

Total answered questions: 405


In [37]:
# do the same as the above cell but for average number of questions with answers per topic across all users
average_questions_per_topic = question_topic_counts.copy()
average_questions_per_topic['average_per_user'] = average_questions_per_topic['count'] / total_users if total_users > 0 else 0
print("\nAverage questions WITH answers per topic across all users:")
print(average_questions_per_topic[['topic_name', 'count', 'average_per_user']].to_string(index=False))



Average questions WITH answers per topic across all users:
                                                    topic_name  count  average_per_user
                                      Transfer Function Models     73              14.6
                      Theoretical Models of Chemical Processes     70              14.0
   Dynamic Behaviour of First-Order and Second-Order Processes     68              13.6
                               Introduction to Process Control     66              13.2
Dynamic Response Characteristics of More Complicated Processes     65              13.0
                                            Laplace Transforms     63              12.6


In [38]:
# filter by user 202
user_id = 202
# Get conversations for user 202
user_202_conv_ids = conversations_df[conversations_df['user_id'] == user_id]['id'].unique()
# Get messages from those conversations
user_202_message_ids = messages_df[messages_df['conversation_id'].isin(user_202_conv_ids)]['id'].unique()
# Get questions from those messages
user_202_question_ids = questions_df[questions_df['message_id'].isin(user_202_message_ids)]['id'].unique()
# Get answered questions
user_202_answered_questions = answers_df[answers_df['question_id'].isin(user_202_question_ids)]

print(f"\nUser {user_id} has {len(user_202_answered_questions)} answered questions out of {len(user_202_question_ids)} total questions asked.")



User 202 has 38 answered questions out of 38 total questions asked.


In [39]:
# Split User 202 analysis by topic
user_id = 202

# Get User 202's answered questions with topics
user_202_answered_questions_topics = question_topics_df[question_topics_df['question_id'].isin(user_202_answered_questions['question_id'])]

# Count by topic
user_202_topic_counts = user_202_answered_questions_topics['topic_id'].value_counts().reset_index()
user_202_topic_counts.columns = ['topic_id', 'count']

# Merge with topic names
user_202_topic_counts = user_202_topic_counts.merge(topics_df[['id', 'topic_name']], left_on='topic_id', right_on='id')
user_202_topic_counts = user_202_topic_counts[['topic_name', 'count']].sort_values('count', ascending=False)

print(f"\nUser {user_id} - Questions WITH answers by topic:")
print(user_202_topic_counts.to_string(index=False))
print(f"\nTotal answered questions for User {user_id}: {len(user_202_answered_questions_topics)}")

# Compare with average
print(f"\n{'='*100}")
print(f"Comparison with All Users Average:")
print(f"{'='*100}")
comparison = user_202_topic_counts.merge(
    average_questions_per_topic[['topic_name', 'average_per_user']], 
    on='topic_name', 
    how='left'
)
comparison['difference'] = comparison['count'] - comparison['average_per_user']
comparison['percentage_vs_avg'] = ((comparison['count'] / comparison['average_per_user']) - 1) * 100
comparison = comparison.sort_values('count', ascending=False)

print(comparison.to_string(index=False))



User 202 - Questions WITH answers by topic:
                                                    topic_name  count
                                      Transfer Function Models     18
                      Theoretical Models of Chemical Processes     16
   Dynamic Behaviour of First-Order and Second-Order Processes     14
Dynamic Response Characteristics of More Complicated Processes     13
                                            Laplace Transforms     11
                               Introduction to Process Control      9

Total answered questions for User 202: 81

Comparison with All Users Average:
                                                    topic_name  count  average_per_user  difference  percentage_vs_avg
                                      Transfer Function Models     18              14.6         3.4          23.287671
                      Theoretical Models of Chemical Processes     16              14.0         2.0          14.285714
   Dynamic Behaviour of First

In [40]:
# Test the new backend endpoint for topic comparison
import requests

# Start the backend if needed, then test the endpoint
API_URL = "http://localhost:5002"  # Adjust if your backend runs on a different port
user_id = 202

try:
    response = requests.post(f"{API_URL}/api/topic-comparison", json={"user_id": user_id})
    if response.status_code == 200:
        data = response.json()
        print(f"✓ Topic comparison data retrieved for user {data['user_id']}")
        print(f"Total users in system: {data['total_users']}\n")
        print("Comparison data (suitable for reflective bar chart):")
        print("="*100)
        for item in data['comparison']:
            print(f"{item['topic_name'][:60]:60} | User: {item['user_count']:2} | Avg: {item['average_per_user']:5.2f} | Diff: {item['difference']:+6.2f} | % vs Avg: {item['percentage_vs_avg']:+6.2f}%")
    else:
        print(f"Error: {response.status_code} - {response.text}")
except Exception as e:
    print(f"Error connecting to backend: {str(e)}")
    print("\nNote: Make sure the backend server is running on the specified port.")
    print("You can start it with: python -m dashboard_backend.api_server")


Error connecting to backend: HTTPConnectionPool(host='localhost', port=5002): Max retries exceeded with url: /api/topic-comparison (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x00000160C4317770>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))

Note: Make sure the backend server is running on the specified port.
You can start it with: python -m dashboard_backend.api_server


In [41]:
# conversation metrics - calculate average duration of conversations across all users
# Duration is calculated from first to last message timestamp in each conversation

all_durations = []
for conv_id in conversations_df['id']:
    conv_messages = messages_df[messages_df['conversation_id'] == conv_id]
    if len(conv_messages) > 1:  # Need at least 2 messages to calculate duration
        conv_messages['timestamp'] = pd.to_datetime(conv_messages['timestamp'])
        first_msg = conv_messages['timestamp'].min()
        last_msg = conv_messages['timestamp'].max()
        duration_minutes = (last_msg - first_msg).total_seconds() / 60
        all_durations.append(duration_minutes)

if len(all_durations) > 0:
    avg_duration = sum(all_durations) / len(all_durations)
    print(f"Average conversation duration across all users: {avg_duration:.2f} minutes")
    print(f"Total conversations analyzed: {len(all_durations)}")
    print(f"Median duration: {sorted(all_durations)[len(all_durations)//2]:.2f} minutes")
    print(f"Min duration: {min(all_durations):.2f} minutes")
    print(f"Max duration: {max(all_durations):.2f} minutes")
else:
    print("No conversations with multiple messages found")


Average conversation duration across all users: 25.07 minutes
Total conversations analyzed: 45
Median duration: 25.15 minutes
Min duration: 12.18 minutes
Max duration: 38.90 minutes


C:\Users\garre\AppData\Local\Temp\ipykernel_36632\2358915773.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  conv_messages['timestamp'] = pd.to_datetime(conv_messages['timestamp'])
C:\Users\garre\AppData\Local\Temp\ipykernel_36632\2358915773.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  conv_messages['timestamp'] = pd.to_datetime(conv_messages['timestamp'])
C:\Users\garre\AppData\Local\Temp\ipykernel_36632\2358915773.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice f

In [42]:
# number of questions per question category (solo taxonomy) for user 202
user_id = 202

# Get user 202's questions
user_questions = questions_df[questions_df['id'].isin(user_202_question_ids)]

# Count by solo taxonomy level
solo_counts_user = user_questions['solo_taxonomy_level'].value_counts().sort_values(ascending=False)

print(f"\nUser {user_id} - Questions per Category (Solo Taxonomy):")
print(solo_counts_user)
print(f"Total questions: {len(user_questions)}")



User 202 - Questions per Category (Solo Taxonomy):
solo_taxonomy_level
Unistructural        11
Relational           10
Multistructural       9
Extended Abstract     8
Name: count, dtype: int64
Total questions: 38


In [43]:
# number of questions per question category (solo taxonomy) for the average user
# Get all questions and count by solo taxonomy
all_solo_counts = questions_df['solo_taxonomy_level'].value_counts().sort_values(ascending=False)
avg_per_user = all_solo_counts / total_users

print(f"\nAverage Questions per Category (Solo Taxonomy) across all users:")
print(f"Total counts: {all_solo_counts.to_dict()}")
print(f"\nAverage per user:")
for category, avg in avg_per_user.items():
    print(f"  {category}: {avg:.2f}")

# Comparison
print(f"\n\nComparison: User {user_id} vs All Users Average")
print("=" * 60)
comparison_solo = pd.DataFrame({
    'category': solo_counts_user.index,
    'user_202_count': solo_counts_user.values
})
comparison_solo['average_per_user'] = comparison_solo['category'].map(avg_per_user)
comparison_solo['difference'] = comparison_solo['user_202_count'] - comparison_solo['average_per_user']
comparison_solo = comparison_solo.sort_values('user_202_count', ascending=False)

print(comparison_solo.to_string(index=False))



Average Questions per Category (Solo Taxonomy) across all users:
Total counts: {'Relational': 53, 'Unistructural': 52, 'Multistructural': 51, 'Extended Abstract': 43}

Average per user:
  Relational: 10.60
  Unistructural: 10.40
  Multistructural: 10.20
  Extended Abstract: 8.60


Comparison: User 202 vs All Users Average
         category  user_202_count  average_per_user  difference
    Unistructural              11              10.4         0.6
       Relational              10              10.6        -0.6
  Multistructural               9              10.2        -1.2
Extended Abstract               8               8.6        -0.6


In [44]:
# mean answer accuracy score per question category for user 202
user_id = 202

# Get user 202's questions and their answers
user_questions_with_answers = questions_df[questions_df['id'].isin(user_202_question_ids)].copy()

# Merge with answers to get accuracy scores
user_questions_with_answers = user_questions_with_answers.merge(
    answers_df[['question_id', 'accuracy_score']], 
    left_on='id', 
    right_on='question_id', 
    how='left'
)

# Calculate mean accuracy score by solo taxonomy level
mean_accuracy_user = user_questions_with_answers.groupby('solo_taxonomy_level')['accuracy_score'].agg(['mean', 'count'])
mean_accuracy_user = mean_accuracy_user.sort_values('mean', ascending=False)

print(f"\nUser {user_id} - Mean Answer Accuracy Score per Category (Solo Taxonomy):")
print(mean_accuracy_user)



User 202 - Mean Answer Accuracy Score per Category (Solo Taxonomy):
                          mean  count
solo_taxonomy_level                  
Extended Abstract    55.250000      8
Relational           49.500000     10
Multistructural      47.777778      9
Unistructural        29.909091     11


In [ ]:
# mean answer accuracy score per question category for all users (average)
# Get all questions with answers
all_questions_with_answers = questions_df.copy()
all_questions_with_answers = all_questions_with_answers.merge(
    answers_df[['question_id', 'accuracy_score']], 
    left_on='id', 
    right_on='question_id', 
    how='left'
)

# Calculate mean accuracy score by solo taxonomy level
mean_accuracy_all = all_questions_with_answers.groupby('solo_taxonomy_level')['accuracy_score'].agg(['mean', 'count'])
mean_accuracy_all = mean_accuracy_all.sort_values('mean', ascending=False)

print(f"\nAverage Mean Answer Accuracy Score per Category (Solo Taxonomy):")
print(mean_accuracy_all)

# Comparison: User 202 vs All Users
print(f"\n\nComparison: User {user_id} vs All Users Average")
print("=" * 70)
comparison_accuracy = pd.DataFrame({
    'category': mean_accuracy_user.index,
    'user_202_mean': mean_accuracy_user['mean'].values,
    'user_202_count': mean_accuracy_user['count'].values
})
comparison_accuracy['all_users_mean'] = comparison_accuracy['category'].map(mean_accuracy_all['mean'])
comparison_accuracy['difference'] = comparison_accuracy['user_202_mean'] - comparison_accuracy['all_users_mean']
comparison_accuracy = comparison_accuracy.sort_values('user_202_mean', ascending=False)

print(comparison_accuracy.to_string(index=False))


Average Mean Answer Accuracy Score per Category (Solo Taxonomy):
                          mean  count
solo_taxonomy_level                  
Multistructural      52.235294     51
Unistructural        51.961538     52
Relational           49.358491     53
Extended Abstract    46.697674     43


Comparison: User 202 vs All Users Average
         category  user_202_mean  user_202_count  all_users_mean  all_users_count  difference
Extended Abstract      55.250000               8       46.697674               43    8.552326
       Relational      49.500000              10       49.358491               53    0.141509
  Multistructural      47.777778               9       52.235294               51   -4.457516
    Unistructural      29.909091              11       51.961538               52  -22.052448


In [ ]:
# Format accuracy comparison data for the dashboard chart
# This transforms the comparison_accuracy dataframe into chart-ready format

# For the frontend "Answer Accuracy per Question Category" chart
chart_data = {
    'individual_data': comparison_accuracy[['category', 'user_202_mean']].rename(columns={'category': 'solo_category', 'user_202_mean': 'mean_accuracy'}).to_dict('records'),
    'average_data': comparison_accuracy[['category', 'all_users_mean']].rename(columns={'category': 'solo_category', 'all_users_mean': 'mean_accuracy'}).to_dict('records'),
    'full_comparison': comparison_accuracy.to_dict('records')
}

print("Chart data for 'Answer Accuracy per Question Category':")
print("\nUser 202 Accuracy by Category:")
for item in chart_data['individual_data']:
    print(f"  {item['solo_category']}: {item['mean_accuracy']:.2f}")

print("\nClass Average Accuracy by Category:")
for item in chart_data['average_data']:
    print(f"  {item['solo_category']}: {item['mean_accuracy']:.2f}")

print("\nFull Comparison Table:")
print(comparison_accuracy.to_string(index=False))